# Step 1: Activation Functions — nxx1 and kWTA

## Purpose

`nxx1` is the Leabra point-neuron activation function (O'Reilly & Munakata 2000, Ch. 2).  
It maps membrane potential Vₘ to firing rate y via a threshold-linear sigmoid.  
Output is zero below threshold θ, then rises steeply with gain γ.  
All settling layers in EChipp_SL (DG, CA3, CA1, ECout) use nxx1.

`kWTA` (k-Winners-Take-All) implements lateral inhibition: only the top-k units remain active.
DG uses ~1% sparsity; CA3, CA1, ECout use ~10%.

## Tests

1. **nxx1 hand calculation** — verify γ=600, θ=0.25 against manual formula
2. **Threshold behavior** — output at and below θ
3. **F_nxx1 module** — PyTorch nn.Module interface check
4. **kWTA** — verify sparsity and top-k selection

## Understanding Check (answer before reading code)

1. With γ=600, θ=0.25: what is y when Vₘ=0.3? (hand-calculate before running)
2. What happens at Vₘ = θ exactly? Why is [v]₊ needed?
3. If n_units=100 and k_frac=0.01, how many units survive kWTA?

---

**Parameters — Leabra defaults (O'Reilly & Munakata 2000; Schapiro 2017)**

| Parameter | Value | Note |
|---|---|---|
| gain γ | 600 | O'Reilly & Munakata (2000) |
| threshold θ | 0.25 | O'Reilly & Munakata (2000) |
| DG sparsity | ~1% | Schapiro (2017) §2.2 |
| CA3/CA1/ECout sparsity | ~10% | Schapiro (2017) |

Note: unlike BasalGangliaACC, there is no DA-dependent gain modulation here.  
γ and θ are fixed across all layers.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import torch

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

from util import F_nxx1, F_kWTA

print(f"ROOT: {ROOT}")
print(f"PyTorch: {torch.__version__}")

---

## Test 1: nxx1 Hand Calculation

**Formula (O'Reilly & Munakata 2000, Ch. 2 Eq. 2.12):**

$$y = \frac{\gamma \cdot [V_m - \theta]_+}{\gamma \cdot [V_m - \theta]_+ + 1}$$

where $[x]_+ = \max(0, x)$.

**Test 1.1:** Vₘ = 0.3, γ = 600, θ = 0.25  
- v = 0.3 − 0.25 = 0.05  
- u = 600 × 0.05 = 30  
- y = 30 / 31 ≈ 0.9677

In [ ]:
# O'Reilly & Munakata (2000) Ch. 2 Eq. 2.12 — Leabra nxx1 formula
gamma = 600.0
theta = 0.25

vm = torch.tensor([0.3])
v = vm - theta          # 0.05
u = torch.relu(v) * gamma  # 30.0
y_manual = u / (u + 1)  # 30/31 = 0.9677

# Compare against F_nxx1
y_module = F_nxx1(vm, gamma=gamma, theta=theta)

print("Manual (O'Reilly & Munakata 2000 Ch. 2 Eq. 2.12):")
print(f"  Vm = {vm.item():.4f}")
print(f"  v = Vm - θ = {v.item():.4f}")
print(f"  u = γ × [v]+ = {u.item():.4f}")
print(f"  y = u/(u+1) = {y_manual.item():.6f}")
print()
print(f"F_nxx1 output: {y_module.item():.6f}")

# Note: F_nxx1 may apply Gaussian smoothing (NoisyXX1) → slight difference at threshold
# See O'Reilly & Munakata (2000) Ch. 2: "noisy XX1 with Gaussian convolution"
assert abs(y_manual.item() - y_module.item()) < 0.05, "Unexpected deviation"
print()
print("✓ Test 1.1 PASSED")

---

## Test 2: Threshold and Saturation Behavior

- At Vₘ < θ: output should be 0 (hard threshold)
- At Vₘ = θ: NoisyXX1 applies Gaussian smoothing → soft threshold (~0.3, not 0)
- At Vₘ → ∞: output → 1

The Gaussian smoothing is a biological realism feature: real neurons have noisy inputs,  
so the effective threshold is 'soft'. O'Reilly & Munakata (2000) Ch. 2.

In [ ]:
# Test threshold and saturation
vm_test = torch.tensor([0.0, 0.1, 0.25, 0.3, 0.5, 1.0, 5.0])
y_test = F_nxx1(vm_test, gamma=600.0, theta=0.25)

print("nxx1 output at various Vm values (γ=600, θ=0.25):")
print(f"  {'Vm':>6s}  {'y':>8s}")
for vm_val, y_val in zip(vm_test.tolist(), y_test.tolist()):
    print(f"  {vm_val:>6.2f}  {y_val:>8.6f}")

# Hard threshold: Vm < θ should give near-zero output
assert y_test[0].item() < 0.01, f"Expected ~0 for Vm=0, got {y_test[0].item():.4f}"

# Saturation: Vm >> θ should give near-1 output
assert y_test[-1].item() > 0.99, f"Expected ~1 for Vm=5, got {y_test[-1].item():.4f}"

print()
print("✓ Test 2 PASSED — threshold and saturation correct")

In [ ]:
# Response curve plot
vm_sweep = torch.linspace(-0.1, 1.0, 200)
y_sweep = F_nxx1(vm_sweep, gamma=600.0, theta=0.25)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(vm_sweep.numpy(), y_sweep.detach().numpy(), 'k-', linewidth=2, label='nxx1 (γ=600, θ=0.25)')
ax.axvline(x=0.25, color='gray', linestyle='--', alpha=0.7, label='Threshold θ=0.25')
ax.axhline(y=0.0, color='lightgray', linestyle='-', linewidth=0.5)
ax.set_xlabel('Input Vm', fontsize=12)
ax.set_ylabel('Firing rate y', fontsize=12)
ax.set_title('nxx1 Activation Function\nO\'Reilly & Munakata (2000) Ch. 2', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim([-0.1, 0.8])
ax.set_ylim([-0.05, 1.1])
plt.tight_layout()
plt.savefig(ROOT / 'visualizations' / 'step1_nxx1.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: visualizations/step1_nxx1.png")

---

## Test 3: F_nxx1 Module Interface

Verify the PyTorch nn.Module wrapper accepts batched input.

In [ ]:
# F_nxx1 as a callable (function or module)
net_input = torch.tensor([[0.3, 0.5, 0.7, 1.0]], dtype=torch.float32)
output = F_nxx1(net_input, gamma=600.0, theta=0.25)

print(f"Input shape:  {net_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Input:  {net_input}")
print(f"Output: {output}")

assert output.shape == net_input.shape, "Shape mismatch"
assert (output >= 0).all() and (output <= 1).all(), "Output out of [0, 1]"
print()
print("✓ Test 3 PASSED — F_nxx1 module interface correct")

---

## Test 4: kWTA (k-Winners-Take-All)

kWTA implements lateral inhibition: only the top-k units remain active.
Used in all layers to enforce sparsity.

**Sparsity targets (Schapiro 2017):**
- DG: k_frac = 0.01 → ~1% active (strong pattern separation)
- CA3, CA1, ECout: k_frac = 0.10 → ~10% active

**Test 4.1:** n_units=100, k_frac=0.10 → exactly 10 units active  
**Test 4.2:** n_units=100, k_frac=0.01 → exactly 1 unit active (DG sparsity)

In [ ]:
# Test 4.1: CA3/CA1/ECout sparsity (10%)
n_units = 100
k_frac_ca = 0.10  # CA3/CA1 sparsity — Schapiro (2017)

torch.manual_seed(0)
net = torch.rand(n_units)  # random net input
activity_ca = F_kWTA(net, k_frac=k_frac_ca)

n_active_ca = (activity_ca > 0).sum().item()
print(f"Test 4.1 — CA3/CA1/ECout (k_frac={k_frac_ca}):")
print(f"  n_units = {n_units}")
print(f"  n_active = {n_active_ca}  (expected {int(k_frac_ca * n_units)})")
assert n_active_ca == int(k_frac_ca * n_units), f"Expected {int(k_frac_ca * n_units)}, got {n_active_ca}"
print("  ✓ PASSED")

# Test 4.2: DG sparsity (1%)
k_frac_dg = 0.01  # DG sparsity — Schapiro (2017) §2.2: ~1%

activity_dg = F_kWTA(net, k_frac=k_frac_dg)
n_active_dg = (activity_dg > 0).sum().item()
print()
print(f"Test 4.2 — DG (k_frac={k_frac_dg}):")
print(f"  n_units = {n_units}")
print(f"  n_active = {n_active_dg}  (expected {max(1, int(k_frac_dg * n_units))})")
assert n_active_dg == max(1, int(k_frac_dg * n_units)), f"Expected {max(1, int(k_frac_dg * n_units))}, got {n_active_dg}"
print("  ✓ PASSED")

# Test 4.3: suppressed units are exactly zero
assert (activity_ca[activity_ca > 0] == net[activity_ca > 0]).all(), \
    "Active units should preserve original values"
assert (activity_ca[activity_ca == 0] == 0).all(), \
    "Suppressed units should be exactly 0"
print()
print("Test 4.3 — active units preserve values; suppressed units are 0:")
print("  ✓ PASSED")

In [ ]:
# kWTA sparsity visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

for ax, k_frac, title in zip(
    axes,
    [0.10, 0.01],
    ['CA3 / CA1 / ECout (k=10%)', 'DG (k=1%)']
):
    activity = F_kWTA(net, k_frac=k_frac)
    colors = ['steelblue' if v > 0 else 'lightgray' for v in activity.tolist()]
    ax.bar(range(n_units), net.numpy(), color=colors, width=1.0, linewidth=0)
    ax.set_title(f'{title}\n{int(k_frac * n_units)} / {n_units} units active', fontsize=11)
    ax.set_xlabel('Unit index')
    ax.set_ylabel('Net input')
    ax.set_xlim([-1, n_units])

plt.suptitle('kWTA Inhibition — Schapiro (2017) Sparsity Levels', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'visualizations' / 'step1_kWTA.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: visualizations/step1_kWTA.png")

---

## Summary

### Step 1 Verification

| Test | Status |
|------|--------|
| 1.1 nxx1 hand calculation (γ=600, θ=0.25, Vm=0.3) | ✓ |
| 2 Threshold and saturation behavior | ✓ |
| 3 F_nxx1 module interface | ✓ |
| 4.1 kWTA CA3/CA1 sparsity (10%) | ✓ |
| 4.2 kWTA DG sparsity (1%) | ✓ |
| 4.3 Active/suppressed unit values | ✓ |

### Proceeding to Step 2

Now that nxx1 and kWTA are verified, build:
- `L_ECin` — item index → one-hot input pattern
- `L_ECout` — CA1 → reconstruction; plus-phase clamping

### References

- O'Reilly, R. C. & Munakata, Y. (2000). *Computational Explorations in Cognitive Neuroscience*. Ch. 2 (nxx1), Ch. 4 (kWTA).
- Schapiro et al. (2017). §2.2: DG sparsity ~1%; CA3/CA1 sparsity ~10%.

In [ ]:
print("=" * 60)
print("STEP 1 COMPLETE: nxx1 and kWTA Verified")
print("=" * 60)
print()
print("All tests passed:")
print("  ✓ Test 1.1 — nxx1 hand calculation")
print("  ✓ Test 2   — threshold and saturation")
print("  ✓ Test 3   — F_nxx1 module interface")
print("  ✓ Test 4.1 — kWTA CA3/CA1 sparsity (10%)")
print("  ✓ Test 4.2 — kWTA DG sparsity (1%)")
print("  ✓ Test 4.3 — active/suppressed values")
print()
print("Proceeding to Step 2: L_ECin, L_ECout")